# Notebook 12: Parametric catastrophe-bond basis risk

Fit one source-specific magnitude-distance trigger to **I0 training years
1–1,000,000**, freeze it, then evaluate the same payouts against I0, C1, and C2
in **years 1,000,001–2,000,000**. The indemnity target is Notebook 11's frozen
occurrence XoL recovery. No ground motions, building losses, policy terms,
catalog events, or dependence models are regenerated.

This is a synthetic one-year collateralized bond design, with principal equal
to the frozen occurrence limit. Each catalog year represents a new issuance;
principal is depleted in event-time order and resets annually, with no
reinstatement. Nominal event-trigger diagnostics are reported alongside actual
collateralized payments. The indemnity benchmark has no annual recovery cap,
so collateral depletion is explicitly part of the comparison.

Run the setup and input cell first. Production artifacts stay under ignored
`data/processed/`; small portable audit tables and handoffs go under
`data/metadata/phase_2/notebook_12_parametric_basis_risk/`.


In [ ]:
from pathlib import Path, PurePosixPath
import gzip
import hashlib
import io
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "tools" / "reinsurance_capital.py").is_file():
    raise RuntimeError("Start Jupyter from the seismic-correlation-insurance-loss repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools import parametric_basis_risk as parametric_module
from tools.parametric_basis_risk import (
    ATTACHMENT_GRID, DECAY_GRID, STEP_GRID, PAYOUT_FRACTIONS, SOURCES,
    annual_basis_series, apply_annual_collateral, basis_metrics, calibrate_trigger,
    portfolio_distances, predict_payout, tail_rows, validate_features,
)
from tools.reinsurance_capital import (
    CASE_PREFIXES, apply_occurrence_xol, paired_bootstrap_differences,
)

CATALOG_YEARS = 2_000_000
TRAIN_END = 1_000_000
EXPECTED_OCCURRENCES = 10_630
EXPECTED_OCCUPIED_YEARS = 10_593
EXPECTED_SITES = 470
EXPECTED_RUPTURES = 3_927
FROZEN_ATTACHMENT = 18_811_083.54014544
PRINCIPAL = 61_837_983.314918146
ROW_TOLERANCE_USD = 2.0e-6
AGGREGATE_TOLERANCE_USD = 0.01
BOOTSTRAP_REPLICATES = 300
BOOTSTRAP_SEED = 122_026
RETURN_PERIODS = (100, 250, 500, 1_000, 2_500, 5_000, 10_000, 50_000, 100_000, 1_000_000)
META = ROOT / "data/metadata/phase_2/notebook_12_parametric_basis_risk"
PROCESSED = ROOT / "data/processed/phase_2/notebook_12_parametric_basis_risk"
INPUTS = {
    "notebook11_handoff": {
        "path": "data/metadata/phase_2/notebook_11_reinsurance_capital/notebook_11_final_handoff.json",
        "sha256": "c9a9ad70a7eb01d679e227442a2f11667a39900d2246550d71e0bb247d316b1e",
    },
    "catalog": {
        "path": "data/processed/annual_event_catalog/annual_event_catalog.csv",
        "sha256": "e4725a57eab466348aa263d79b8f8bcc923f9542fb5507af279fece41c121f37",
    },
    "distances": {
        "path": "data/processed/notebook_4_authoritative_usgs_distances/usgs_rupture_site_distances.csv.gz",
        "sha256": "dbc5c000adb368b1f2e85ea353073bb00f84ae815ee61311844d10162ae9d719",
    },
    "sites": {
        "path": "data/metadata/notebook_4_cell_18_site_order.csv",
        "sha256": "fc64d53410fdadab5379f82c73c1d74ce3393911fbdd73a0dbc483f056f3d100",
        "hash_mode": "canonical_crlf",
    },
    "parametric_module": {"path": "tools/parametric_basis_risk.py", "sha256": "e3796b2566d807222f103e6ed2e81399832d62ca3f95efb88e1b7156d658a476", "hash_mode": "lf"},
    "reinsurance_module": {"path": "tools/reinsurance_capital.py", "sha256": "6364939c42f8700d1f9d8f2c25d2c8eee7c2facc38cb23cf0bed5da57417353c", "hash_mode": "lf"},
}

def relative(path):
    return Path(path).resolve().relative_to(ROOT).as_posix()

def sha256_file(path, mode="raw"):
    if mode != "raw":
        raw = Path(path).read_bytes().replace(b"\r\n", b"\n").replace(b"\r", b"\n")
        if mode == "canonical_crlf":
            raw = raw.replace(b"\n", b"\r\n")
        elif mode != "lf":
            raise ValueError("Unknown hash normalization.")
        return hashlib.sha256(raw).hexdigest()
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, payload):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_bytes((json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n").encode("utf-8"))

def write_csv(path, frame):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    if str(path).endswith(".gz"):
        with Path(path).open("wb") as raw:
            with gzip.GzipFile(fileobj=raw, mode="wb", filename="", mtime=0) as compressed:
                with io.TextIOWrapper(compressed, encoding="utf-8", newline="") as handle:
                    frame.to_csv(handle, index=False, float_format="%.17g", lineterminator="\n", chunksize=25_000)
    else:
        Path(path).write_bytes(frame.to_csv(index=False, float_format="%.17g", lineterminator="\n").encode("utf-8"))

def read_csv(path, **kwargs):
    return pd.read_csv(path, float_precision="round_trip", **kwargs)

def check(rows, check_id, passed, detail, severity="critical"):
    rows.append(dict(check_id=check_id, severity=severity, passed=bool(passed), detail=str(detail)))

def gate(rows, name):
    frame = pd.DataFrame(rows)
    failed = frame.loc[frame.severity.eq("critical") & ~frame.passed]
    if not failed.empty:
        display(failed)
        raise RuntimeError(name + " failed.")
    return frame

def passed_booleans(series):
    normalized = series.astype(str).str.strip().str.lower()
    if not normalized.isin(["true", "false"]).all():
        raise ValueError("Malformed validation boolean.")
    return normalized.eq("true")

def inventory_item(path):
    return dict(path=relative(path), bytes=Path(path).stat().st_size, sha256=sha256_file(path))

def period_label(years):
    return np.where(np.asarray(years) <= TRAIN_END, "TRAINING", "EVALUATION")

print("Notebook 12 setup complete. Next: validate the frozen inputs.")


## Declared trigger and evaluation contract

For each source type, the index is
$I=M-b\log_{10}(1+R_{\min}/50)$, where $R_{\min}$ is the minimum authoritative
rupture distance in kilometres over the frozen 470-site portfolio. It is not
an epicentral or portfolio-centroid approximation.

The declared synthetic grid is $a=5.00,5.25,\ldots,9.25$,
$b\in\{0.5,1,1.5,2\}$ and tier spacing $s\in\{0.25,0.5,0.75,1\}$.
Crossing $a,a+s,a+2s,a+3s$ pays 25%, 50%, 75%, 100% of principal;
equality enters the higher tier. Below $a$ there is no payout.
Choose the minimum **I0 training-event mean squared nominal basis error**
separately for INTERFACE and SLAB. The first candidate wins an exact tie.
This objective fits nominal event payments before collateral depletion.
These grid choices are experimental assumptions, not externally calibrated
parameters. Evaluation results do not change the grid or the selected terms.

Positive basis is target indemnity recovery minus payout. False-positive and
false-negative probabilities are labeled by denominator; a tolerance of
$0.000002 is used only for classification. Monetary values are not rounded
away. Annual cash net loss is gross loss minus actual payout and can be
negative. Report its positive part as **unfunded loss** and its negative part
as **surplus**; excess payouts are not erased. Event shortfall is summed before
annual aggregation, so excess at another occurrence does not hide it.

All annual denominators include zero-event years. Headline comparisons use
the evaluation half only. Training and full-catalog summaries are diagnostics.
PML uses descending rank `ceil(N/R)`; VaR and TVaR use the frozen Notebook 11
fixed-count estimator `ceil(N*(1-q))`. Tail support below 20 observations is
diagnostic. The bootstrap conditions on the frozen trigger and quantifies
evaluation-catalog sampling uncertainty, not calibration or model uncertainty.
Premium, discounting, counterparty default, and market pricing are outside
this loss-based experiment.


In [ ]:
input_checks = []
for name, item in INPUTS.items():
    path = ROOT / item["path"]
    actual = sha256_file(path, item.get("hash_mode", "raw")) if path.is_file() else "MISSING"
    check(input_checks, name + "_hash", actual == item["sha256"], f"path={item['path']}; sha256={actual}")
gate(input_checks, "Notebook 12 frozen input hashes")
upstream = json.loads((ROOT / INPUTS["notebook11_handoff"]["path"]).read_text(encoding="utf-8"))
controls = upstream["frozen_controls"]
check(input_checks, "notebook11_complete", upstream["notebook11_complete"] and upstream["validation"]["critical_failures"] == 0, "Validated Notebook 11 handoff.")
check(input_checks, "frozen_dimensions", controls["catalog_years"] == CATALOG_YEARS and controls["occurrences"] == EXPECTED_OCCURRENCES and controls["sites"] == EXPECTED_SITES, controls)
check(input_checks, "common_dependence_cases", upstream["dependence_cases"] == list(CASE_PREFIXES), upstream["dependence_cases"])
layer = controls["occurrence_layer"]
check(input_checks, "frozen_occurrence_terms", layer["attachment_2022_usd"] == FROZEN_ATTACHMENT and layer["limit_2022_usd"] == PRINCIPAL and layer["participation"] == 1.0, layer)
for item in upstream["artifact_inventory"]:
    path = ROOT / item["path"]
    matched = path.is_file() and path.stat().st_size == item["bytes"] and sha256_file(path) == item["sha256"]
    check(input_checks, "notebook11_artifact:" + path.name, matched, item["path"])
gate(input_checks, "Notebook 12 upstream artifacts")
prior_validation = read_csv(ROOT / upstream["validation"]["path"])
check(input_checks, "upstream_critical_checks", passed_booleans(prior_validation.loc[prior_validation.severity.eq("critical"), "passed"]).all(), "Read the published validation table.")

events = read_csv(ROOT / upstream["output_contract"]["event_reinsurance"]["path"])
catalog = read_csv(ROOT / INPUTS["catalog"]["path"], usecols=["event_id", "simulation_year", "time_within_year", "rupture_id", "source_type", "magnitude"])
catalog = catalog.rename(columns={"event_id": "catalog_event_id", "simulation_year": "catalog_year"})
check(input_checks, "event_dimensions", len(events) == EXPECTED_OCCURRENCES and events.occurrence_id.is_unique and events.catalog_year.nunique() == EXPECTED_OCCUPIED_YEARS, f"events={len(events)}; occupied={events.catalog_year.nunique()}")
check(input_checks, "catalog_event_identity", len(catalog) == len(events) and catalog.catalog_event_id.is_unique and set(catalog.catalog_event_id) == set(events.catalog_event_id), "One frozen catalog record per occurrence.")
gate(input_checks, "Notebook 12 catalog identity")
aligned_catalog = catalog.set_index("catalog_event_id").loc[events.catalog_event_id].reset_index()
for column in ["catalog_year", "rupture_id", "source_type"]:
    check(input_checks, "catalog_match:" + column, np.array_equal(events[column].to_numpy(), aligned_catalog[column].to_numpy()), "Exact frozen catalog field match.")
magnitude_error = float(np.max(np.abs(events.magnitude.to_numpy() - aligned_catalog.magnitude.to_numpy())))
check(input_checks, "catalog_match:magnitude", magnitude_error <= 1.0e-12, f"maximum_magnitude_error={magnitude_error:.6e}; CSV round-trip tolerance=1e-12")
# Predict with the original catalog magnitude, avoiding chained CSV rounding.
events["magnitude"] = aligned_catalog.magnitude.to_numpy()
events["time_within_year"] = aligned_catalog.time_within_year.to_numpy()

sites = read_csv(ROOT / INPUTS["sites"]["path"])
check(input_checks, "frozen_sites", len(sites) == EXPECTED_SITES and sites.site_id.is_unique, f"sites={len(sites)}")
distances = read_csv(ROOT / INPUTS["distances"]["path"], usecols=["rupture_id", "site_id", "r_rup_km"], dtype={"rupture_id": "category", "site_id": "category"})
rupture_distances = portfolio_distances(distances, sites.site_id)
check(input_checks, "authoritative_distance_dimensions", len(rupture_distances) == EXPECTED_RUPTURES and len(distances) == EXPECTED_RUPTURES * EXPECTED_SITES, f"ruptures={len(rupture_distances)}; pairs={len(distances)}")
del distances
events = events.merge(rupture_distances, on="rupture_id", how="left", validate="many_to_one", sort=False)
validate_features(events, CATALOG_YEARS)
check(input_checks, "complete_source_distance_features", len(events) == EXPECTED_OCCURRENCES and events.portfolio_distance_km.notna().all(), "Minimum authoritative Rrup joined without changing occurrences.")
for prefix in CASE_PREFIXES.values():
    gross = events[f"{prefix}_gross_insured_loss_2022_usd"].to_numpy()
    expected = apply_occurrence_xol(gross, FROZEN_ATTACHMENT, PRINCIPAL)
    error = float(np.max(np.abs(expected["ceded_loss_2022_usd"] - events[f"{prefix}_frozen_occurrence_ceded_loss_2022_usd"])))
    check(input_checks, prefix + "_target_reconstruction", error <= ROW_TOLERANCE_USD, f"maximum_error_2022_usd={error:.6e}")
input_validation = gate(input_checks, "Notebook 12 input validation")
write_csv(META / "notebook_12_input_validation.csv", input_validation)
print("=" * 78)
print("NOTEBOOK 12 CELL 1: FROZEN PARAMETRIC INPUTS VALIDATED")
print("=" * 78)
print(f"Dependence cases:             {len(CASE_PREFIXES)}")
print(f"Catalog occurrences:          {len(events):,}")
print(f"Authoritative rupture rows:   {len(rupture_distances):,}")
print(f"Bond principal:               ${PRINCIPAL:,.2f}")
print(f"Input validation checks:      {len(input_validation)}")
print("Next: fit I0 training years and freeze the trigger.")


In [ ]:
trigger, candidate_grid = calibrate_trigger(
    events, "i0_frozen_occurrence_ceded_loss_2022_usd", PRINCIPAL,
    train_end=TRAIN_END, declared_years=CATALOG_YEARS,
)
TRIGGER_PATH = META / "notebook_12_frozen_trigger.json"
write_json(TRIGGER_PATH, trigger)
FROZEN_TRIGGER_HASH = sha256_file(TRIGGER_PATH)
write_csv(META / "notebook_12_trigger_candidate_grid.csv", candidate_grid)
model_specification = {
    "schema_version": "notebook12_parametric_model_v1",
    "input_provenance": INPUTS,
    "dependence_cases": list(CASE_PREFIXES),
    "training_years": [1, TRAIN_END], "evaluation_years": [TRAIN_END + 1, CATALOG_YEARS],
    "candidate_grid": {"attachment_index": list(ATTACHMENT_GRID), "distance_decay": list(DECAY_GRID), "tier_step": list(STEP_GRID)},
    "payout_fractions": list(PAYOUT_FRACTIONS), "principal_2022_usd": PRINCIPAL,
    "target": "frozen occurrence XoL ceded recovery; no annual recovery cap",
    "collateral": "one-year issuance; annual reset; no reinstatement; allocate by time_within_year then occurrence_id",
    "calibration": "I0 training nominal event squared basis error; annual collateral cap is applied after fitting",
    "basis_sign": "target indemnity recovery minus payout",
    "cash_net": "gross annual loss minus actual payout; surplus retained explicitly",
    "classification_tolerance_2022_usd": ROW_TOLERANCE_USD,
    "monetary_values_clipped_for_classification": False,
    "headline_period": "EVALUATION",
    "return_periods_years": list(RETURN_PERIODS),
    "tail_estimator": "PML: ceil(N/R); VaR and TVaR: ceil(N*(1-q)) largest annual observations, including zero years",
    "bootstrap": {"replicates": BOOTSTRAP_REPLICATES, "seed": BOOTSTRAP_SEED, "scope": "paired evaluation-year sampling conditional on frozen trigger; no refitting"},
    "trigger": inventory_item(TRIGGER_PATH),
}
write_json(META / "notebook_12_model_specification.json", model_specification)
feature_summary = (events.assign(period=period_label(events.catalog_year))
    .groupby(["period", "source_type"], sort=True).agg(
        occurrences=("occurrence_id", "size"), minimum_magnitude=("magnitude", "min"),
        maximum_magnitude=("magnitude", "max"), minimum_distance_km=("portfolio_distance_km", "min"),
        maximum_distance_km=("portfolio_distance_km", "max"))).reset_index()
write_csv(META / "notebook_12_feature_summary.csv", feature_summary)
display(candidate_grid.loc[candidate_grid.selected])
print("=" * 78)
print("NOTEBOOK 12 CELL 2: I0 TRAINING TRIGGER FROZEN")
print("=" * 78)
print(f"Training years:               1-{TRAIN_END:,}")
print(f"Evaluation years:             {TRAIN_END + 1:,}-{CATALOG_YEARS:,}")
print(f"Candidate rows:               {len(candidate_grid)}")
print(f"Trigger SHA256:               {FROZEN_TRIGGER_HASH}")
print("Next: apply identical payments and annual collateral to all cases.")


In [ ]:
if sha256_file(TRIGGER_PATH) != FROZEN_TRIGGER_HASH:
    raise RuntimeError("The frozen trigger changed before evaluation.")
trigger = json.loads(TRIGGER_PATH.read_text(encoding="utf-8"))
nominal_payout = predict_payout(events, trigger)
actual_payout = apply_annual_collateral(events, nominal_payout, PRINCIPAL, declared_years=CATALOG_YEARS)
event_output = events.copy()
event_output["period"] = period_label(events.catalog_year)
event_output["nominal_payout_2022_usd"] = nominal_payout
event_output["collateralized_payout_2022_usd"] = actual_payout
event_output["collateral_depletion_shortfall_2022_usd"] = nominal_payout - actual_payout
for prefix in CASE_PREFIXES.values():
    target = events[f"{prefix}_frozen_occurrence_ceded_loss_2022_usd"].to_numpy()
    event_output[f"{prefix}_signed_basis_2022_usd"] = target - actual_payout
    event_output[f"{prefix}_protection_shortfall_2022_usd"] = np.maximum(target - actual_payout, 0.0)
    event_output[f"{prefix}_excess_payout_2022_usd"] = np.maximum(actual_payout - target, 0.0)
annual = annual_basis_series(events, {"nominal": nominal_payout, "collateralized": actual_payout}, declared_years=CATALOG_YEARS)
EVENT_PATH = PROCESSED / "paired_parametric_event_basis_risk.csv.gz"
ANNUAL_PATH = PROCESSED / "paired_parametric_annual_basis_risk.csv.gz"
write_csv(EVENT_PATH, event_output)
write_csv(ANNUAL_PATH, annual)
print("=" * 78)
print("NOTEBOOK 12 CELL 3: PAIRED PAYOUTS AND ANNUAL SERIES COMPLETE")
print("=" * 78)
print(f"Event rows:                   {len(event_output):,}")
print(f"Annual rows including zeros:  {len(annual):,}")
print(f"Depleted event payments:      {np.count_nonzero(nominal_payout - actual_payout > ROW_TOLERANCE_USD):,}")
print("Next: evaluate basis risk and residual annual tails.")


In [ ]:
periods = {"TRAINING": (1, TRAIN_END), "EVALUATION": (TRAIN_END + 1, CATALOG_YEARS), "FULL_DIAGNOSTIC": (1, CATALOG_YEARS)}
summary_rows, tail_frames = [], []
for period, (first, last) in periods.items():
    mask = events.catalog_year.between(first, last).to_numpy()
    duration = last - first + 1
    subset = events.loc[mask]
    for payout_basis, all_payouts in [("NOMINAL_EVENT", nominal_payout), ("COLLATERALIZED", actual_payout)]:
        payout = all_payouts[mask]
        for case_name, prefix in CASE_PREFIXES.items():
            target = subset[f"{prefix}_frozen_occurrence_ceded_loss_2022_usd"].to_numpy()
            gross = subset[f"{prefix}_gross_insured_loss_2022_usd"].to_numpy()
            metrics = basis_metrics(target, payout, gross, declared_years=duration)
            negative = (target > ROW_TOLERANCE_USD) & (payout <= ROW_TOLERANCE_USD)
            positive = (target <= ROW_TOLERANCE_USD) & (payout > ROW_TOLERANCE_USD)
            metrics["false_negative_probability_per_year"] = subset.loc[negative, "catalog_year"].nunique() / duration
            metrics["false_positive_probability_per_year"] = subset.loc[positive, "catalog_year"].nunique() / duration
            metrics["collateral_depletion_aal_2022_usd"] = float((nominal_payout[mask] - actual_payout[mask]).sum() / duration)
            summary_rows.append(dict(period=period, payout_basis=payout_basis, case_name=case_name, case_prefix=prefix, **metrics))
    tails = tail_rows(annual.iloc[first - 1:last], return_periods=RETURN_PERIODS)
    tails.insert(0, "period", period)
    tails["declared_years"] = duration
    tail_frames.append(tails)
basis_summary = pd.DataFrame(summary_rows)
tail_table = pd.concat(tail_frames, ignore_index=True)
delta_rows = []
metrics_to_compare = ["payout_aal_2022_usd", "target_aal_2022_usd", "expected_protection_shortfall_2022_usd", "expected_excess_payout_2022_usd", "event_rmse_2022_usd"]
for payout_basis, frame in basis_summary.loc[basis_summary.period.eq("EVALUATION")].groupby("payout_basis", sort=True):
    baseline = frame.loc[frame.case_prefix.eq("i0")].iloc[0]
    for row in frame.loc[~frame.case_prefix.eq("i0")].itertuples():
        for name in metrics_to_compare:
            value, control = float(getattr(row, name)), float(baseline[name])
            delta_rows.append(dict(period="EVALUATION", payout_basis=payout_basis, case_name=row.case_name, metric=name, independent_value_2022_usd=control, correlated_value_2022_usd=value, absolute_change_2022_usd=value-control, percent_change=100*(value/control-1) if control else np.nan))
paired_deltas = pd.DataFrame(delta_rows)
write_csv(META / "notebook_12_basis_risk_summary.csv", basis_summary)
write_csv(META / "notebook_12_tail_risk_table.csv", tail_table)
write_csv(META / "notebook_12_paired_case_differences.csv", paired_deltas)
display(basis_summary.loc[basis_summary.period.eq("EVALUATION") & basis_summary.payout_basis.eq("COLLATERALIZED"), ["case_name", *metrics_to_compare]])
print("NOTEBOOK 12 CELL 4: OUT-OF-SAMPLE BASIS RISK AND TAIL METRICS COMPLETE")


In [ ]:
evaluation_annual = annual.iloc[TRAIN_END:]
uncertainty_frames = []
shortfall_series = {}
shortfall_pairs = []
for kind in ("shortfall", "excess"):
    for prefix in CASE_PREFIXES.values():
        shortfall_series[f"{prefix}_{kind}"] = evaluation_annual[f"{prefix}_{kind}_sum_2022_usd"].to_numpy()
    shortfall_pairs.extend([(f"{p}_{kind}", f"i0_{kind}") for p in ("c1", "c2")])
uncertainty_frames.append(paired_bootstrap_differences(shortfall_series, shortfall_pairs, [{"name": "annual_mean", "kind": "mean"}], replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED))
tail_series = {}
for prefix in CASE_PREFIXES.values():
    for kind in ("gross", "benchmark_retained", "unfunded"):
        tail_series[f"{prefix}_{kind}"] = evaluation_annual[f"{prefix}_{kind}_aep_2022_usd"].to_numpy()
tail_pairs = [(f"{p}_unfunded", "i0_unfunded") for p in ("c1", "c2")]
tail_pairs += [(f"{p}_unfunded", f"{p}_{kind}") for p in CASE_PREFIXES.values() for kind in ("gross", "benchmark_retained")]
specs = [{"name": "aal", "kind": "mean"}, {"name": "aep_tvar_99_5", "kind": "tvar", "confidence": .995}]
if CATALOG_YEARS - TRAIN_END >= 2_500:
    specs.append({"name": "aep_pml_2500yr", "kind": "pml", "return_period": 2_500})
uncertainty_frames.append(paired_bootstrap_differences(tail_series, tail_pairs, specs, replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED))
uncertainty = pd.concat(uncertainty_frames, ignore_index=True)
uncertainty.insert(0, "period", "EVALUATION")
uncertainty["scope"] = "conditional on frozen trigger; paired evaluation catalog years; no recalibration"
write_csv(META / "notebook_12_uncertainty_summary.csv", uncertainty)
print(f"NOTEBOOK 12 CELL 5: PAIRED UNCERTAINTY COMPLETE ({len(uncertainty)} rows; {BOOTSTRAP_REPLICATES} replicates)")


In [ ]:
checks = []
check(checks, "input_validation_passed", input_validation.passed.all(), f"checks={len(input_validation)}")
check(checks, "event_annual_dimensions", len(events) == EXPECTED_OCCURRENCES and len(annual) == CATALOG_YEARS and annual.catalog_occurrence_count.sum() == len(events), f"events={len(events)}; years={len(annual)}")
check(checks, "all_zero_event_years_preserved", (annual.catalog_occurrence_count == 0).sum() == CATALOG_YEARS - EXPECTED_OCCUPIED_YEARS and (annual.loc[annual.catalog_occurrence_count.eq(0)].iloc[:, 2:] == 0).all().all(), "Zero-event years have zero losses and payouts.")
check(checks, "train_evaluation_partition", feature_summary.occurrences.sum() == len(events) and set(feature_summary.period) == {"TRAINING", "EVALUATION"}, f"split={TRAIN_END}; disjoint catalog-year sets")
check(checks, "complete_common_candidate_grid", len(candidate_grid) == 2*len(ATTACHMENT_GRID)*len(DECAY_GRID)*len(STEP_GRID) and candidate_grid.groupby("source_type").selected.sum().eq(1).all(), f"candidates={len(candidate_grid)}")
check(checks, "trigger_file_unchanged", sha256_file(TRIGGER_PATH) == FROZEN_TRIGGER_HASH, FROZEN_TRIGGER_HASH)
# Repeat fitting using physically training-only rows to catch accidental leakage.
training_only_spec, _ = calibrate_trigger(events.loc[events.catalog_year <= TRAIN_END], "i0_frozen_occurrence_ceded_loss_2022_usd", PRINCIPAL, train_end=TRAIN_END, declared_years=CATALOG_YEARS)
check(checks, "calibration_uses_training_only", training_only_spec == trigger, "Training-only replay yields exactly the saved specification.")
loss_free = events[["occurrence_id", "catalog_year", "source_type", "magnitude", "portfolio_distance_km"]].copy()
check(checks, "prediction_independent_of_losses_and_case", np.array_equal(predict_payout(loss_free, trigger), nominal_payout), "One source-only payout vector is shared across I0, C1, C2.")
check(checks, "payout_and_collateral_bounds", np.all(actual_payout >= 0) and np.all(actual_payout <= nominal_payout) and np.all(nominal_payout <= PRINCIPAL) and annual.collateralized_payout_aep_2022_usd.max() <= PRINCIPAL + ROW_TOLERANCE_USD, f"maximum annual payout={annual.collateralized_payout_aep_2022_usd.max():.8f}")
reverse = np.arange(len(events)-1, -1, -1)
check(checks, "collateral_independent_of_row_order", np.array_equal(apply_annual_collateral(events.iloc[reverse], nominal_payout[reverse], PRINCIPAL, declared_years=CATALOG_YEARS), actual_payout[reverse]), "Frozen event time controls depletion.")
for prefix in CASE_PREFIXES.values():
    gross = annual[f"{prefix}_gross_aep_2022_usd"].to_numpy()
    net = annual[f"{prefix}_cash_net_aep_2022_usd"].to_numpy()
    unfunded = annual[f"{prefix}_unfunded_aep_2022_usd"].to_numpy()
    surplus = annual[f"{prefix}_surplus_aep_2022_usd"].to_numpy()
    payout = annual.collateralized_payout_aep_2022_usd.to_numpy()
    error = float(np.max(np.abs(gross + surplus - payout - unfunded)))
    check(checks, prefix + "_cash_accounting", error <= ROW_TOLERANCE_USD and np.all(unfunded >= 0) and np.all(surplus >= 0) and np.array_equal(net, unfunded-surplus), f"maximum_error_2022_usd={error:.6e}")
    target = events[f"{prefix}_frozen_occurrence_ceded_loss_2022_usd"].to_numpy()
    basis = event_output[f"{prefix}_signed_basis_2022_usd"].to_numpy()
    event_error = float(np.max(np.abs(basis - (target - actual_payout))))
    total_errors = [abs(events[f"{prefix}_gross_insured_loss_2022_usd"].sum() - gross.sum()), abs(target.sum() - annual[f"{prefix}_target_aep_2022_usd"].sum()), abs(actual_payout.sum() - payout.sum())]
    control_row = next(row for row in upstream["fixed_program_results"] if row["case_prefix"] == prefix and row["program"] == "FROZEN_OCCURRENCE_XOL")
    aal_error = abs(annual[f"{prefix}_target_aep_2022_usd"].mean() - control_row["ceded_aal_2022_usd"])
    check(checks, prefix + "_event_annual_upstream_reconciliation", event_error <= ROW_TOLERANCE_USD and max(total_errors) <= AGGREGATE_TOLERANCE_USD and aal_error <= ROW_TOLERANCE_USD, f"event_error={event_error:.6e}; total_error={max(total_errors):.6e}; upstream_aal_error={aal_error:.6e}")
check(checks, "basis_summary_complete", len(basis_summary) == 18 and basis_summary.groupby(["period", "payout_basis"]).payout_aal_2022_usd.nunique().eq(1).all(), "Three periods, two payout bases, three cases; identical payout AAL across cases.")
pml = tail_table.loc[tail_table.metric.eq("aep_pml")]
ordered = all(group.sort_values("parameter").value_2022_usd.diff().dropna().ge(-ROW_TOLERANCE_USD).all() for _, group in pml.groupby(["period", "case_prefix", "loss_basis"]))
check(checks, "pml_monotonicity_and_tail_support", ordered and np.array_equal(tail_table.headline_supported, tail_table.tail_count >= 20), f"rows={len(tail_table)}")
check(checks, "paired_evaluation_uncertainty", len(uncertainty) == 4 + 8*len(specs) and uncertainty.bootstrap_replicates.eq(BOOTSTRAP_REPLICATES).all() and uncertainty.period.eq("EVALUATION").all(), f"rows={len(uncertainty)}; replicates={BOOTSTRAP_REPLICATES}")
public_paths = [item["path"] for item in INPUTS.values()] + [relative(META), relative(PROCESSED), relative(TRIGGER_PATH)]
check(checks, "public_paths_are_repository_relative", all(not PurePosixPath(p).is_absolute() and ".." not in PurePosixPath(p).parts and "\\" not in p and ":" not in p for p in public_paths), "POSIX paths relative to the repository.")
check(checks, "extreme_return_period_tail_support", tail_table.headline_supported.all(), "Order-statistic support below 20 is diagnostic, not a headline estimate.", "warning")
gross_var = tail_table.loc[tail_table.period.eq("EVALUATION") & tail_table.loss_basis.eq("gross") & tail_table.metric.eq("var")]
check(checks, "var_capital_informative", gross_var.value_2022_usd.gt(0).all(), "A zero VaR under sparse losses is non-informative; report TVaR and PML alongside it.", "warning")
check(checks, "calibration_uncertainty_included", False, "Paired bootstrap conditions on the frozen trigger; calibration and model uncertainty are outside these intervals.", "warning")
final_validation = pd.DataFrame(checks)
write_csv(META / "notebook_12_final_validation.csv", final_validation)
gate(checks, "Notebook 12 full-catalog validation")
display(final_validation)
print("NOTEBOOK 12 CELL 6: ALL CRITICAL VALIDATION CHECKS PASSED")


In [ ]:
gate(checks, "Notebook 12 final handoff gate")
public_names = [
    "notebook_12_input_validation.csv", "notebook_12_model_specification.json",
    "notebook_12_feature_summary.csv", "notebook_12_frozen_trigger.json",
    "notebook_12_trigger_candidate_grid.csv", "notebook_12_basis_risk_summary.csv",
    "notebook_12_tail_risk_table.csv", "notebook_12_paired_case_differences.csv",
    "notebook_12_uncertainty_summary.csv", "notebook_12_final_validation.csv",
]
inventory = [inventory_item(META / name) for name in public_names]
inventory += [inventory_item(EVENT_PATH), inventory_item(ANNUAL_PATH)]
handoff = {
    "schema_version": "notebook12_parametric_basis_risk_handoff_v1",
    "pipeline_version": "notebook12_parametric_basis_risk_v1",
    "notebook12_complete": True,
    "upstream": INPUTS["notebook11_handoff"],
    "frozen_controls": {"phase1_release": controls["phase1_release"], "phase1_commit": controls["phase1_commit"], "catalog_years": CATALOG_YEARS, "occurrences": len(events), "sites": EXPECTED_SITES, "training_years": [1, TRAIN_END], "evaluation_years": [TRAIN_END + 1, CATALOG_YEARS], "occurrence_attachment_2022_usd": FROZEN_ATTACHMENT, "bond_principal_2022_usd": PRINCIPAL},
    "dependence_cases": list(CASE_PREFIXES),
    "trigger": inventory_item(TRIGGER_PATH),
    "headline_period": "EVALUATION",
    "output_contract": {"event_basis_risk": {"path": relative(EVENT_PATH), "rows": len(event_output)}, "annual_basis_risk": {"path": relative(ANNUAL_PATH), "rows": len(annual)}, "basis_summary": {"path": relative(META / "notebook_12_basis_risk_summary.csv"), "rows": len(basis_summary)}, "tail_table": {"path": relative(META / "notebook_12_tail_risk_table.csv"), "rows": len(tail_table)}},
    "basis_sign": "target indemnity recovery minus actual parametric payout",
    "limitations": ["synthetic candidate grid; I0 training only", "one-year principal cap differs from uncapped annual occurrence recoveries", "signed cash net loss, unfunded loss, and surplus are reported separately", "rank below 20 is diagnostic", "bootstrap conditional on frozen trigger; not calibration or model uncertainty", "loss analysis without premium, discounting, or market pricing"],
    "artifact_inventory": inventory,
    "validation": {"path": relative(META / "notebook_12_final_validation.csv"), "sha256": sha256_file(META / "notebook_12_final_validation.csv"), "checks": len(final_validation), "critical_failures": 0, "warnings": int((final_validation.severity.eq("warning") & ~final_validation.passed).sum())},
    "next_notebook": "13_phase_2_results_and_validation.ipynb",
    "next_task": "Combine validated Phase 2 findings, evaluation-only basis-risk comparisons, uncertainty and limitations into final tables, figures and release validation.",
}
write_json(META / "notebook_12_final_handoff.json", handoff)
print("=" * 78)
print("NOTEBOOK 12 COMPLETE: PARAMETRIC BASIS RISK VALIDATED")
print("=" * 78)
print(f"Dependence cases:             {len(CASE_PREFIXES)}")
print(f"Training/evaluation years:    {TRAIN_END:,} / {CATALOG_YEARS - TRAIN_END:,}")
print(f"Trigger candidate rows:       {len(candidate_grid)}")
print(f"Basis-risk summary rows:      {len(basis_summary)}")
print(f"Validation checks:            {len(final_validation)}")
print("Critical failures:            0")
print(f"Final handoff:                {relative(META / 'notebook_12_final_handoff.json')}")
print("Next: Notebook 13 Phase 2 results and validation.")


## Reading the results

Use the EVALUATION rows for performance statements. The nominal event rows
isolate trigger mismatch; the collateralized rows include finite annual
principal. Identical payouts can still produce different shortfall, excess,
and unfunded tails because the indemnity targets vary with spatial dependence.
The case-difference table gives the independent value, correlated value,
absolute change, and percentage change. Undefined ratios and constant-series
correlations are blank, not fabricated zeros.

Small cash-accounting differences are checked against the frozen numerical
tolerance. Substantive surplus is retained and never treated as a numerical
error. Do not infer that a zero 99.5% VaR means there is no catastrophe risk;
read TVaR, PML, and their tail counts. Sampling intervals condition on the
fitted trigger and omit refitting uncertainty.

Notebook 13 will consolidate the results and release validation. It will not
silently recalibrate this trigger after inspecting the evaluation sample.
